In [6]:
import pandas as pd
import requests
import json
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon 



In [8]:
demographics = json.loads(requests.get('https://files.jcrayb.com/files/ie300/majority.json').text)

In [9]:
chicago = gpd.read_file('osmnx/data/zipcodes.geojson').dropna(subset='zip')[['zip', 'geometry']]

In [10]:
centroids = {}

tracts = gpd.read_file('osmnx/data/tracts.geojson')



tract_demographics = {}

zips = chicago.zip.to_list()

zips = [zip for zip in zips if zip]

demographics = {zip: majority for zip, majority in demographics.items() if zip in zips}

for tract in tracts.iloc:
    centroid = tract.geometry.centroid

    for row in chicago.iloc:
        zip_geometry = row.geometry

        if zip_geometry.contains(centroid) and (row.zip in demographics):
            tract_demographics[tract.namelsad10] = demographics[row.zip]
            continue


In [11]:
json.dump(tract_demographics, open('data/tract_demographics.json', 'w'), indent=2)

In [12]:
query = 'hospital'

destination = json.load(open(f'computation_results/budget_paths/{query}_final_results.json', 'r'))

n = 1

## Includes no cameras

In [13]:


df = pd.DataFrame(index=list(tract_demographics.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])



for tract, race in tract_demographics.items():
    #print(tract, race)
    sorted_results = sorted(destination[tract], key=lambda x: list(destination[tract][x].items())[0][1])
    
    first_n_res = sorted_results[:n]
    n_cameras = []

    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for place in first_n_res:
        results = destination[tract][place]
        #print(results)

        budgets = list(results.keys())
        highest, lowest = (budgets[0], budgets[-1])
        #print(highest, lowest)

        n_cameras += [int(highest)]
        unrestricted_time_to_reach += [results[highest]]
        restricted_time_to_reach += [results[lowest]]
    
    mean_uttr = np.mean(unrestricted_time_to_reach)
    mean_rttr = np.mean(restricted_time_to_reach)

    df.loc[df.index == tract] = [race, mean_uttr, mean_rttr, (mean_rttr-mean_uttr)/mean_uttr, mean_rttr-mean_uttr, np.mean(n_cameras)]

df.to_csv(f'./analysis/{query}-{n}-closest.csv')

## Only compares with cameras

In [14]:
df = pd.DataFrame(index=list(tract_demographics.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])

for tract, race in tract_demographics.items():
    print(tract, race)
    sorted_results = sorted(destination[tract], key=lambda x: list(destination[tract][x].items())[0][1])
    
    first_n_res = sorted_results[:n]
    n_cameras = []

    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for place in first_n_res:
        results = destination[tract][place]
        

        budgets = list(results.keys())
        highest, lowest = (budgets[0], budgets[-1])
        if highest == lowest: continue
        #print(highest, lowest)
        print(results)
        n_cameras += [int(highest)]
        unrestricted_time_to_reach += [results[highest]]
        restricted_time_to_reach += [results[lowest]]
    
    mean_uttr = np.mean(unrestricted_time_to_reach)
    mean_rttr = np.mean(restricted_time_to_reach)

    df.loc[df.index == tract] = [race, mean_uttr, mean_rttr, (mean_rttr-mean_uttr)/mean_uttr, mean_rttr-mean_uttr, np.mean(n_cameras)]

df.to_csv(f'./analysis/{query}-{n}-closest-nonzero.csv')

Census Tract 8424 black
Census Tract 8403 white
{'1': 339.20000000000005, '0': 339.7}
Census Tract 8411 white
Census Tract 8412 white
Census Tract 8390 white
Census Tract 8382 black
Census Tract 6503.01 white
{'1': 206.4, '0': 215.8}
Census Tract 5305.03 black
Census Tract 7608.03 white
Census Tract 306.01 white
Census Tract 306.04 white
{'1': 70.5, '0': 251.0}
Census Tract 208.01 white
Census Tract 5401.02 black
Census Tract 8433 white
Census Tract 5401.01 black
Census Tract 4402.01 black
{'1': 250.3, '0': 282.0}
Census Tract 802.02 white
Census Tract 701.02 white
Census Tract 315.01 white
Census Tract 315.02 white
Census Tract 8349 black
Census Tract 8348 black
Census Tract 1605.02 white
Census Tract 1407.02 white
Census Tract 8420 white
Census Tract 1504.02 white
{'2': 141.7, '1': 164.1}
Census Tract 8344 black
Census Tract 402.01 white
Census Tract 402.02 white
Census Tract 207.02 white
Census Tract 208.02 white
{'1': 111.30000000000001, '0': 121.2}
Census Tract 203.01 white
Census

/home/jcrayb/Documents/GitHub/transport-research-project/lib64/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jcrayb/Documents/GitHub/transport-research-project/lib64/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [15]:
total_results

NameError: name 'total_results' is not defined

In [ ]:
query = 'hospital'

destination = json.load(open(f'computation_results/budget_paths/{query}_final_results.json', 'r'))

t = destination['Census Tract 8424']

t

{'Jackson Park Hospital & Medical Center': {'2': 415.0,
  '1': 420.29999999999995,
  '0': 431.6},
 'South Shore Hospital': {'2': 503.8, '1': 530.3999999999999, '0': 532.6},
 'Chicago Hospital': {'1': 500.70000000000005, '0': 502.09999999999997},
 'St. Bernard Hospital': {'0': 322.59999999999997},
 'Advocate Trinity Hospital': {'2': 532.1, '1': 544.8000000000001, '0': 605.9},
 'South Ashland Medical Center': {'2': 512.7,
  '1': 524.0999999999999,
  '0': 524.0999999999999},
 'La Rabida Children’s Hospital': {'1': 607.1999999999999, '0': 614.9},
 'UChicago Medicine Center for Care and Discovery - Hyde Park': {'1': 504.8,
  '0': 513.1},
 'LCMH - Mt. Greenwood Medical Center': {'1': 684.2, '0': 803.3},
 'U of C hospital': {'1': 504.8, '0': 513.1},
 'Mitchell Hospital': {'1': 509.3, '0': 510.69999999999993},
 'Holy Cross Hospital Specialty Clinic': {'1': 665.0, '0': 683.5999999999999},
 "Comer Children's Hospital": {'1': 500.70000000000005,
  '0': 502.09999999999997},
 'Provident Hospital of

In [54]:
import plotly.graph_objects as go
fig = go.Figure()

fig.add_trace(go.Scattermap(
        lon=[-87.63004035576937],
        lat=[41.74247518248426],
        mode='markers',
        marker=go.scattermap.Marker(
            size=10,
            color='rgb(0, 0, 255)',
            opacity=0.7
        ),
        text=1,
        hoverinfo='text'
    ))

hospitals = json.load(open(f'computation_results/{query}.json', 'r'))

lat = [h[0] for k, h in list(hospitals['Census Tract 8424'].items())]
lon = [h[1] for k, h in list(hospitals['Census Tract 8424'].items())]

fig.add_trace(go.Scattermap(
        lon=lon,
        lat=lat,
        mode='markers',
        marker=go.scattermap.Marker(
            size=10,
            color='rgb(255, 0, 0)',
            opacity=0.7
        ),
        text=1,
        hoverinfo='text'
    ))



fig.update_layout(
    map = {
        #'style': "open-street-map",
        'center': { 'lon': -87.63004035576937, 'lat': 41.74247518248426},
        'zoom': 12},
    margin = {'l':0, 'r':0, 'b':0, 't':0})

fig.update_layout(height=1000, width=1000)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'lat': [41.74247518248426],
              'lon': [-87.63004035576937],
              'marker': {'color': 'rgb(0, 0, 255)', 'opacity': 0.7, 'size': 10},
              'mode': 'markers',
              'text': '1',
              'type': 'scattermap'},
             {'hoverinfo': 'text',
              'lat': [41.7578041, 41.7493665, 41.7901233, 41.7785287, 41.7263401,
                      41.7499017, 41.7774763, 41.79098, 41.6934265, 41.7917659,
                      41.7892031, 41.7679664, 41.7902873, 41.8027413, 41.7936265,
                      41.7212559, 41.7260083, 41.766016],
              'lon': [-87.585201, -87.5689169, -87.6055479, -87.6329532,
                      -87.5670825, -87.6845933, -87.5715858, -87.6049447,
                      -87.7005954, -87.6050931, -87.6041142, -87.6908246,
                      -87.604476, -87.613229, -87.5841361, -87.7320491,
                      -87.5677084, -87.5732189],
              'marker': {'color': 'rgb(255, 0, 0)', 'opacity': 0.7, 'size': 10},
              'mode': 'markers',
              'text': '1',
              'type': 'scattermap'}],
    'layout': {'height': 1000,
               'map': {'center': {'lat': 41.74247518248426, 'lon': -87.63004035576937}, 'zoom': 12},
               'margin': {'b': 0, 'l': 0, 'r': 0, 't': 0},
               'template': '...',
               'width': 1000}
})

In [53]:
tracts = gpd.read_file('osmnx/data/tracts.geojson')

g = [i for i in list(tracts.iloc[0].geometry)]

all_coords = []
for b in g.boundary: # for first feature/row
    coords = np.dstack(b.coords.xy).tolist()
    all_coords.append(*coords) 

TypeError: 'MultiPolygon' object is not iterable

In [63]:
tracts.iloc[0].geometry.boundary.coords

NotImplementedError: Sub-geometries may have coordinate sequences, but multi-part geometries do not

In [1]:
import pandas as pd

In [19]:
data = pd.read_csv('data/location.csv')

data.loc[(data.loctype == 1) | (data.loctype == 2) | (data.loctype == 3)]

,sampno,locno,perno,loctype,state,country,state_fips,county_fips,tract_fips,out_region,home,latitude,longitude
0,20000083,10000,-1,1,IL,USA,17,31,802403,0,1,42.141870,-87.953126
1,20000083,10002,1,2,IL,USA,17,31,801601,0,0,42.145289,-87.863023
11,20000136,10000,-1,1,IL,USA,17,31,81402,0,1,41.891022,-87.612931
12,20000136,10002,1,2,IL,USA,17,31,830900,0,0,41.928424,-87.684906
17,20000136,20002,2,2,IL,USA,17,31,81401,0,0,41.895035,-87.619717
...,...,...,...,...,...,...,...,...,...,...,...,...,...
110072,70100992,10002,1,2,IL,USA,17,97,864205,0,0,42.292549,-88.141232
110074,70100993,10000,-1,1,IL,USA,17,31,260700,0,1,41.877056,-87.727996
110075,70100993,20002,2,2,IL,USA,17,43,844401,0,0,41.828246,-88.016876
110082,70100998,10000,-1,1,IL,USA,17,31,841400,0,1,41.853339,-87.714393
